In [ ]:
#Оценка LSTM модели на задаче генерации текста с использованием ROUGE метрики

import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

# Путь к папке src, где лежит lstm_model.py
src_path = os.path.join(notebook_dir, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from lstm_model import BiRNNClassifier


import torch
from transformers import BertTokenizerFast
from lstm_model import BiRNNClassifier
import os
from tqdm import tqdm
import evaluate
import numpy as np

# Загружаем метрику ROUGE
rouge_metric = evaluate.load("rouge")


def generate_and_evaluate(model, tokenizer, text, device):
    """
    Генерация и оценка текста:
    1️⃣ Один токен
    2️⃣ 1/4 текста
    """
    tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(tokens) < 2:
        return None

    # Контекст для генерации: 3/4 текста
    context_len = int(len(tokens) * 0.75)
    context_one_token = tokens[:context_len]
    target_quarter = tokens[context_len:]

    # 1️⃣ Определяем target_one как первый «реальный» токен из оставшейся четверти
    target_one_token = None
    for tok in target_quarter:
        if tok not in {tokenizer.sep_token_id, tokenizer.cls_token_id, tokenizer.pad_token_id, tokenizer.cls_token_id}:
            target_one_token = [tok]
            break
    if target_one_token is None:
        target_one_token = []

    # Генерация одного токена
    generated_one = model.generate_text(
        context_one_token,
        max_new_tokens=1,
        temperature=1.0,
        device=device,
        tokenizer=tokenizer
    )
    generated_one_token = generated_one[-1:]  # последний токен

    # 2️⃣ Генерация 1/4 текста, начиная с сгенерированного токена
    # context_quarter = context_one_token + generated_one_token
    remaining_len = len(target_quarter)  
    generated_quarter = model.generate_text(
        context_one_token,
        max_new_tokens=remaining_len,
        temperature=1,
        device=device,
        tokenizer=tokenizer
    )
    # Берём только сгенерированное продолжение четверти текста
    generated_quarter_only = generated_one_token + generated_quarter[len(context_one_token):]

    # Декодирование
    original_text = tokenizer.decode(tokens, skip_special_tokens=True)
    context_text = tokenizer.decode(context_one_token, skip_special_tokens=True)
    gen_one_text = tokenizer.decode(generated_one_token, skip_special_tokens=True)
    gen_quarter_text = tokenizer.decode(generated_quarter_only, skip_special_tokens=True)
    target_one_text = tokenizer.decode(target_one_token, skip_special_tokens=True)
    target_quarter_text = tokenizer.decode(target_quarter, skip_special_tokens=True)

    # ROUGE
    rouge_one = rouge_metric.compute(predictions=[gen_one_text], references=[target_one_text])
    rouge_quarter = rouge_metric.compute(predictions=[gen_quarter_text], references=[target_quarter_text])

    return {
        "original": original_text,
        "context": context_text,
        "generated_one": gen_one_text,
        "generated_quarter": gen_quarter_text,
        "target_one": target_one_text,
        "target_quarter": target_quarter_text,
        "rouge_one": rouge_one,
        "rouge_quarter": rouge_quarter
    }



def evaluate_texts(model, tokenizer, texts, device, print_examples=True):
    rouge_one_scores = []
    rouge_quarter_scores = []

    # Печать первых 10 примеров
    if print_examples:
        print("\n=== ПЕРВЫЕ 10 ТЕСТОВЫХ ПРИМЕРОВ ===")
        for i, text in enumerate(texts[20:30]):
            res = generate_and_evaluate(model, tokenizer, text, device)
            if res is None:
                continue

            print(f"\nПример {i+1}:")
            print(f"Оригинальный текст: {res['original']}")
            print(f"Промт (3/4 текста): {res['context']}")
            print(f"Сгенерировано один токен: {res['generated_one']} (правильный: {res['target_one']})")
            print(f"Сгенерировано 1/4 текста: {res['generated_quarter']} (правильное продолжение: {res['target_quarter']})")
            print(f"ROUGE-1 один токен: {res['rouge_one']['rouge1']:.4f}")
            print(f"ROUGE-1 1/4 текста: {res['rouge_quarter']['rouge1']:.4f}")
            print("-" * 80)

    # Усреднение ROUGE по всем текстам
    for text in tqdm(texts[:210], desc="Calculating average ROUGE"):
        res = generate_and_evaluate(model, tokenizer, text, device)
        if res is None:
            continue
        rouge_one_scores.append(res['rouge_one']['rouge1'])
        rouge_quarter_scores.append(res['rouge_quarter']['rouge1'])

    avg_rouge_one = np.mean(rouge_one_scores) if rouge_one_scores else 0.0
    avg_rouge_quarter = np.mean(rouge_quarter_scores) if rouge_quarter_scores else 0.0

    print("\n=== СРЕДНИЙ ROUGE ===")
    print(f"Средний ROUGE-1 для одного токена: {avg_rouge_one:.4f}")
    print(f"Средний ROUGE-1 для 1/4 текста: {avg_rouge_quarter:.4f}")


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Путь к модели и токенизатору
    model_path = "models/birnn_lstm_lm.pth"
    tokenizer_path = "data/processed/tokenizer"

    # Загружаем токенизатор
    tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path)

    # Загружаем модель
    checkpoint = torch.load(model_path, map_location=device)
    model = BiRNNClassifier(
        vocab_size=checkpoint['vocab_size'],
        hidden_dim=checkpoint['hidden_dim']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    # Загружаем тестовые тексты
    project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

    # Путь к файлу с тестами
    data_dir = os.path.join(project_root, "data", "processed")
    test_file = os.path.join(data_dir, "test.txt")
    with open(test_file, "r", encoding="utf-8") as f:
        test_texts = [line.strip() for line in f if line.strip()]

    # Вызываем функцию оценки
    evaluate_texts(model, tokenizer, test_texts, device, print_examples=True)


if __name__ == "__main__":
    main()



=== ПЕРВЫЕ 10 ТЕСТОВЫХ ПРИМЕРОВ ===

Пример 1:
Оригинальный текст: sigh am sitting here working with my leg propped up it s making my ankle feel better but also making my knee hurt
Промт (3/4 текста): sigh am sitting here working with my leg propped up it s making my ankle feel better
Сгенерировано один токен: practice (правильный: but)
Сгенерировано 1/4 текста: practice (правильное продолжение: but also making my knee hurt)
ROUGE-1 один токен: 0.0000
ROUGE-1 1/4 текста: 0.0000
--------------------------------------------------------------------------------

Пример 2:
Оригинальный текст: feeling really sick today how about you
Промт (3/4 текста): feeling really sick today how
Сгенерировано один токен: cold (правильный: about)
Сгенерировано 1/4 текста: cold you did talking (правильное продолжение: about you)
ROUGE-1 один токен: 0.0000
ROUGE-1 1/4 текста: 0.3333
--------------------------------------------------------------------------------

Пример 3:
Оригинальный текст: bored amp tire

Calculating average ROUGE: 100%|██████████| 210/210 [01:09<00:00,  3.03it/s]


=== СРЕДНИЙ ROUGE ===
Средний ROUGE-1 для одного токена: 0.0857
Средний ROUGE-1 для 1/4 текста: 0.0354


In [4]:
#оценка трансформерной модели на задаче генерации текста с использованием ROUGE метрики GPT2 вместо distilgpt2, так как distilgpt2 давал точность оклоло 2% что меньше чем LSTM модель

import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import evaluate
import numpy as np
from tqdm import tqdm


import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

# Путь к папке src, где лежит lstm_model.py
src_path = os.path.join(notebook_dir, "src")

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
# --- Функция для загрузки тестовых текстов ---
def load_texts(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
with open(test_file, "r", encoding="utf-8") as f:
    test_texts = [line.strip() for line in f if line.strip()]

# --- Загрузка модели и токенизатора ---
model_name = "GPT2" # на модели distilgpt2 rouge около 2% то есть меньше чем на LSTM поэтому переделал на GPT2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# --- Pipeline для генерации ---
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,  # CPU; для GPU ставьте 0
)

# --- Метрика ROUGE ---
rouge = evaluate.load("rouge")

# --- Функция для генерации и оценки ---
def generate_and_evaluate_transformer(text):
    words = text.split()
    if len(words) < 5 or len(words) > 100:
        return None

    # Делим на 3/4 и 1/4
    split_idx = int(len(words) * 0.75)
    prompt_text = ' '.join(words[:split_idx])
    target_text = ' '.join(words[split_idx:])

    # Генерация продолжения
    out = generator(
        prompt_text,
        max_new_tokens=len(words) - split_idx,
        num_return_sequences=1,
        do_sample=False,
        top_p=0.95,
        temperature=1
    )

    generated_full = out[0]["generated_text"]
    # Берём только сгенерированное продолжение
    if generated_full.startswith(prompt_text):
        generated_part = generated_full[len(prompt_text):].strip()
    else:
        generated_part = generated_full

    # ROUGE
    rouge_scores = rouge.compute(predictions=[generated_part], references=[target_text])

    return {
        "original": text,
        "prompt": prompt_text,
        "generated_quarter": generated_part[:100],
        "target_quarter": target_text,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"]
    }

# --- 1️⃣ Вывод первых 10 примеров ---
print("\n=== ПЕРВЫЕ 10 ПРИМЕРОВ ===")
for i, text in tqdm(enumerate(test_texts[:10])):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue

    print(f"\nПример {i+1}:")
    print(f"Оригинальный текст: {res['original']}")
    print(f"Промт (3/4 текста): {res['prompt']}")
    print(f"Сгенерировано 1/4 текста: {res['generated_quarter']}")
    print(f"Правильное продолжение: {res['target_quarter']}")
    print(f"ROUGE-1: {res['rouge1']:.4f}")
    print(f"ROUGE-2: {res['rouge2']:.4f}")
    print(f"ROUGE-L: {res['rougeL']:.4f}")
    print("-" * 80)

# --- 2️⃣ Средний ROUGE по 100 тестовым текстам ---
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for text in tqdm(test_texts[:100], desc="Вычисление среднего ROUGE"):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue
    rouge1_scores.append(res['rouge1'])
    rouge2_scores.append(res['rouge2'])
    rougeL_scores.append(res['rougeL'])

avg_rouge1 = np.mean(rouge1_scores) if rouge1_scores else 0.0
avg_rouge2 = np.mean(rouge2_scores) if rouge2_scores else 0.0
avg_rougeL = np.mean(rougeL_scores) if rougeL_scores else 0.0

print("\n=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===")
print(f"Средний ROUGE-1: {avg_rouge1:.4f}")
print(f"Средний ROUGE-2: {avg_rouge2:.4f}")
print(f"Средний ROUGE-L: {avg_rougeL:.4f}")


Device set to use cpu



=== ПЕРВЫЕ 10 ПРИМЕРОВ ===


0it [00:00, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
1it [00:00,  2.12it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Пример 1:
Оригинальный текст: she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for what reason but i kinda miss hannah
Промт (3/4 текста): she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for
Сгенерировано 1/4 текста: sure but i think she was just
Правильное продолжение: what reason but i kinda miss hannah
ROUGE-1: 0.2857
ROUGE-2: 0.1667
ROUGE-L: 0.2857
--------------------------------------------------------------------------------


2it [00:00,  2.99it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...


Пример 9:
Оригинальный текст: i dreamt that i was failing one of my classes it was not fun
Промт (3/4 текста): i dreamt that i was failing one of my classes
Сгенерировано 1/4 текста: . I was so
Правильное продолжение: it was not fun
ROUGE-1: 0.2857
ROUGE-2: 0.0000
ROUGE-L: 0.2857
--------------------------------------------------------------------------------


10it [00:02,  3.72it/s]



Пример 10:
Оригинальный текст: thanks i ll need it i hate packing
Промт (3/4 текста): thanks i ll need it i
Сгенерировано 1/4 текста: will be
Правильное продолжение: hate packing
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


Вычисление среднего ROUGE:   0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   1%|          | 1/100 [00:00<00:32,  3.01it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   2%|▏         | 2/100 [00:00<00:27,  3.58it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   3%|▎         | 3/100 [00:00<00:28,  3.39it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` fo


=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===
Средний ROUGE-1: 0.1213
Средний ROUGE-2: 0.0317
Средний ROUGE-L: 0.1213


In [5]:
#оценка трансформерной модели на задаче генерации текста с использованием ROUGE метрики rouge на distilgpt2

import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import evaluate
import numpy as np
from tqdm import tqdm


import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

# Путь к папке src, где лежит lstm_model.py
src_path = os.path.join(notebook_dir, "src")

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
# --- Функция для загрузки тестовых текстов ---
def load_texts(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
with open(test_file, "r", encoding="utf-8") as f:
    test_texts = [line.strip() for line in f if line.strip()]

# --- Загрузка модели и токенизатора ---
model_name = "distilgpt2" # на модели distilgpt2 rouge около 2% то есть меньше чем на LSTM поэтому переделал на GPT2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# --- Pipeline для генерации ---
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,  # CPU; для GPU ставьте 0
)

# --- Метрика ROUGE ---
rouge = evaluate.load("rouge")

# --- Функция для генерации и оценки ---
def generate_and_evaluate_transformer(text):
    words = text.split()
    if len(words) < 5 or len(words) > 100:
        return None

    # Делим на 3/4 и 1/4
    split_idx = int(len(words) * 0.75)
    prompt_text = ' '.join(words[:split_idx])
    target_text = ' '.join(words[split_idx:])

    # Генерация продолжения
    out = generator(
        prompt_text,
        max_new_tokens=len(words) - split_idx,
        num_return_sequences=1,
        do_sample=False,
        top_p=0.95,
        temperature=1
    )

    generated_full = out[0]["generated_text"]
    # Берём только сгенерированное продолжение
    if generated_full.startswith(prompt_text):
        generated_part = generated_full[len(prompt_text):].strip()
    else:
        generated_part = generated_full

    # ROUGE
    rouge_scores = rouge.compute(predictions=[generated_part], references=[target_text])

    return {
        "original": text,
        "prompt": prompt_text,
        "generated_quarter": generated_part[:100],
        "target_quarter": target_text,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"]
    }

# --- 1️⃣ Вывод первых 10 примеров ---
print("\n=== ПЕРВЫЕ 10 ПРИМЕРОВ ===")
for i, text in tqdm(enumerate(test_texts[:10])):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue

    print(f"\nПример {i+1}:")
    print(f"Оригинальный текст: {res['original']}")
    print(f"Промт (3/4 текста): {res['prompt']}")
    print(f"Сгенерировано 1/4 текста: {res['generated_quarter']}")
    print(f"Правильное продолжение: {res['target_quarter']}")
    print(f"ROUGE-1: {res['rouge1']:.4f}")
    print(f"ROUGE-2: {res['rouge2']:.4f}")
    print(f"ROUGE-L: {res['rougeL']:.4f}")
    print("-" * 80)

# --- 2️⃣ Средний ROUGE по 100 тестовым текстам ---
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for text in tqdm(test_texts[:100], desc="Вычисление среднего ROUGE"):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue
    rouge1_scores.append(res['rouge1'])
    rouge2_scores.append(res['rouge2'])
    rougeL_scores.append(res['rougeL'])

avg_rouge1 = np.mean(rouge1_scores) if rouge1_scores else 0.0
avg_rouge2 = np.mean(rouge2_scores) if rouge2_scores else 0.0
avg_rougeL = np.mean(rougeL_scores) if rougeL_scores else 0.0

print("\n=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===")
print(f"Средний ROUGE-1: {avg_rouge1:.4f}")
print(f"Средний ROUGE-2: {avg_rouge2:.4f}")
print(f"Средний ROUGE-L: {avg_rougeL:.4f}")


Device set to use cpu



=== ПЕРВЫЕ 10 ПРИМЕРОВ ===


0it [00:00, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
1it [00:00,  2.79it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Пример 1:
Оригинальный текст: she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for what reason but i kinda miss hannah
Промт (3/4 текста): she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for
Сгенерировано 1/4 текста: sure i dont know for sure i
Правильное продолжение: what reason but i kinda miss hannah
ROUGE-1: 0.1429
ROUGE-2: 0.0000
ROUGE-L: 0.1429
--------------------------------------------------------------------------------

Пример 2:
Оригинальный текст: ahhhhh so when are you leaving will you not make friday
Промт (3/4 текста): ahhhhh so when are you leaving will you
Сгенерировано 1/4 текста: be able to
Правильное продолжение: not make friday
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


2it [00:00,  3.71it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
3it [00:00,  3.73it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...


Пример 8:
Оригинальный текст: the car has to saty with dr toyota overnight
Промт (3/4 текста): the car has to saty with
Сгенерировано 1/4 текста: the car.
Правильное продолжение: dr toyota overnight
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


9it [00:02,  4.49it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
10it [00:02,  4.73it/s]


Пример 9:
Оригинальный текст: i dreamt that i was failing one of my classes it was not fun
Промт (3/4 текста): i dreamt that i was failing one of my classes
Сгенерировано 1/4 текста: .
Правильное продолжение: it was not fun
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------

Пример 10:
Оригинальный текст: thanks i ll need it i hate packing
Промт (3/4 текста): thanks i ll need it i
Сгенерировано 1/4 текста: ll need
Правильное продолжение: hate packing
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


10it [00:02,  4.29it/s]
Вычисление среднего ROUGE:   0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   1%|          | 1/100 [00:00<00:26,  3.73it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   2%|▏         | 2/100 [00:00<00:22,  4.42it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   3%|▎         | 3/100 [00:00<00:22,  4.26it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFO


=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===
Средний ROUGE-1: 0.1025
Средний ROUGE-2: 0.0332
Средний ROUGE-L: 0.1010



Вывод: любая трансформерная модель предпочительнее, чем LSTM  